# Milestone 2: LoRA fine-tuning, checkpoint selection, before/after, final test
Kaggle: **GPU T4 x2, Internet ON**. Total ~45-60 min.

In [6]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one GPU: simpler, reproducible (no DataParallel)
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [7]:
!rm -rf trial-eligibility-transformer
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q tests

Cloning into 'trial-eligibility-transformer'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 64 (delta 16), reused 62 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 52.79 KiB | 1.88 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for trialmatch (pyproject.toml) ... done
...........                                                              [100%]
11 passed in 2.78s


In [8]:
# Rebuilds the identical split (deterministic) and checks it against the manifest in the repo
!python scripts/prepare_data.py
!git diff --stat data/splits/split_manifest.json && echo 'manifest unchanged = split reproduced exactly'

raw rows: 1015  columns: ['annotation_id', 'patient_id', 'note', 'trial_id', 'trial_title', 'criterion_type', 'criterion_text', 'gpt4_explanation', 'explanation_correctness', 'gpt4_sentences', 'expert_sentences', 'gpt4_eligibility', 'expert_eligibility', 'training']

leakage checks: {"patient_overlap": 0, "duplicate_note_across_splits": 0, "annotations_assigned": 1015, "annotations_total": 1015}
dropped rows: {"empty_criterion_text": 1}

split  patients  annots  incl  excl empty_ev
train        37     602   215   387      348
val           8     206    66   140      116
test          8     206    75   131      145

label counts per split:
  train: {'included': 88, 'not included': 13, 'excluded': 8, 'not excluded': 291, 'not enough information': 174, 'not applicable': 28}
  val: {'included': 35, 'not included': 3, 'excluded': 2, 'not excluded': 101, 'not enough information': 53, 'not applicable': 12}
  test: {'included': 27, 'not included': 7, 'excluded': 0, 'not excluded': 91, 'not eno

## 1. Zero-shot baseline on validation (the 'before' model)

In [9]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --split val --tag baseline_qwen25_1p5b

I0000 00:00:1793877747.308776     453 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
baseline_qwen25_1p5b:val: 100%|█████████████████| 26/26 [00:42<00:00,  1.65s/it]

n=206  acc=0.189  macroF1=0.143  invalid=0  | evidence microF1=0.185  macroF1(all)=0.379  macroF1(nonempty gold)=0.256
inclusion acc=0.500  exclusion acc=0.043
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 479, 'mean_prompt_tokens': 372.2330097087379}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/baseline_qwen25_1p5b


## 2. LoRA fine-tuning (train split only; val loss each epoch)

In [10]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less memory fragmentation
!python scripts/train_lora.py

I0000 00:00:1793877803.674487     478 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
train examples: 602 (skipped 0 over max_seq_len)  val: 206  max len: 667  mean len: 374  mean target tokens: 16.4
trainable params: 18,464,768 / 1,562,179,072 (1.18%)
{'loss': 0.2879, 'grad_norm': 1.8133515119552612, 'learning_rate': 0.0001666666666666667, 'epoch': 0.13}
{'loss': 0.1874, 'grad_norm': 0.49220791459083557, 'learning_rate': 0.00019928469263418374, 'epoch': 0.27}
{'loss': 0.1862, 'grad_norm': 0.5272849202156067, 'learning_rate': 0.00019639628606958533, 'epoch': 0.4}
{'loss': 0.1541, 'grad_norm': 0.5192136168479919, 'learning_rate': 0.0001913545457642601, 'epoch': 0.53}
{'loss': 0.1299, 'grad_norm': 0.3991752564907074, 'learning_rate': 0.0001842720958654039, 'epoch': 0.66}
{'loss': 0.123, 'grad_norm': 0.37808215618133545, 'learning_rate': 0.00017530714660036112, 'epoch': 0.8}
{'loss': 0.1355, 'grad_norm': 0.4815822243690491, 'learning_rate': 0.00016465

## 3. Evaluate every epoch checkpoint on validation, pick the best (by val macro-F1)

In [11]:
import glob, subprocess
ckpts = sorted(glob.glob('outputs/lora_qwen25_1p5b/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
assert ckpts, 'no checkpoints found: the training cell above failed, fix that first'
tags = []
for i, ck in enumerate(ckpts, 1):
    tag = f'ft_ep{i}_val'; tags.append(tag)
    subprocess.run(['python', 'scripts/evaluate.py', '--adapter', ck, '--split', 'val', '--tag', tag], check=True)
subprocess.run(['python', 'scripts/select_checkpoint.py', '--run', 'outputs/lora_qwen25_1p5b', '--tags', *tags], check=True)

I0000 00:00:1793878679.068759     508 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ft_ep1_val:val: 100%|██████████| 26/26 [00:36<00:00,  1.40s/it]



n=206  acc=0.733  macroF1=0.468  invalid=0  | evidence microF1=0.323  macroF1(all)=0.311  macroF1(nonempty gold)=0.679
inclusion acc=0.742  exclusion acc=0.729
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 479, 'mean_prompt_tokens': 372.2330097087379}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/ft_ep1_val


I0000 00:00:1793878741.255996     533 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ft_ep2_val:val: 100%|██████████| 26/26 [00:41<00:00,  1.59s/it]



n=206  acc=0.728  macroF1=0.357  invalid=0  | evidence microF1=0.273  macroF1(all)=0.326  macroF1(nonempty gold)=0.713
inclusion acc=0.742  exclusion acc=0.721
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 479, 'mean_prompt_tokens': 372.2330097087379}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/ft_ep2_val


I0000 00:00:1793878808.391670     558 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ft_ep3_val:val: 100%|██████████| 26/26 [00:37<00:00,  1.43s/it]



n=206  acc=0.723  macroF1=0.461  invalid=0  | evidence microF1=0.363  macroF1(all)=0.481  macroF1(nonempty gold)=0.713
inclusion acc=0.712  exclusion acc=0.729
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 479, 'mean_prompt_tokens': 372.2330097087379}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/ft_ep3_val
epoch 1: acc=0.733 macroF1=0.468 evF1=0.323  <- selected
epoch 2: acc=0.728 macroF1=0.357 evF1=0.273
epoch 3: acc=0.723 macroF1=0.461 evF1=0.363
best adapter -> /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/outputs/lora_qwen25_1p5b_best


CompletedProcess(args=['python', 'scripts/select_checkpoint.py', '--run', 'outputs/lora_qwen25_1p5b', '--tags', 'ft_ep1_val', 'ft_ep2_val', 'ft_ep3_val'], returncode=0)

## 4. FINAL held-out TEST (run once, only after steps 1-3 are done)
Same model, same greedy decoding, before vs after fine-tuning.

In [12]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --split test --final-test --tag baseline_qwen25_1p5b_test
!python scripts/evaluate.py --adapter outputs/lora_qwen25_1p5b_best --split test --final-test --tag ft_best_test

I0000 00:00:1793878871.444785     585 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
baseline_qwen25_1p5b_test:test: 100%|███████████| 26/26 [00:38<00:00,  1.46s/it]

n=206  acc=0.170  macroF1=0.156  invalid=0  | evidence microF1=0.203  macroF1(all)=0.418  macroF1(nonempty gold)=0.313
inclusion acc=0.373  exclusion acc=0.053
truncation: {'n_truncated': 0, 'n_gold_evidence_lost': 0, 'max_prompt_tokens': 544, 'mean_prompt_tokens': 341.00970873786406}  parse_failures=0
saved -> /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/baseline_qwen25_1p5b_test
I0000 00:00:1793878924.627241     610 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ft_best_test:test: 100%|████████████████████████| 26/26 [00:34<00:00,  1.31s/it]

n=206  acc=0.728  macroF1=0.465  invalid=0  | evidence microF1=0.342  macroF1(all)=0.243  macroF1(nonempty gold)=0.803
inclusion acc=0.787  exclusion acc=0.695
trunc

## 5. Error analysis + figures

In [13]:
!python scripts/error_analysis.py results/ft_best_test
best = __import__('json').load(open('results/checkpoint_selection.json'))['selected']['tag']
!python scripts/error_analysis.py results/{best}
!python scripts/make_figures.py --train-log outputs/lora_qwen25_1p5b/train_log.json --cm baseline_qwen25_1p5b_test ft_best_test --compare baseline_qwen25_1p5b_test ft_best_test --compare-names 'Zero-shot' 'Fine-tuned (LoRA)'

{
 "inclusion_flip": 0,
 "exclusion_flip": 0,
 "overconfident": 30,
 "missed_information": 14,
 "applicability": 15,
 "invalid_output": 0,
 "evidence_missing": 0,
 "evidence_wrong": 4,
 "evidence_hallucinated": 0
}
8 cases written -> results/ft_best_test/error_analysis.md
{
 "inclusion_flip": 2,
 "exclusion_flip": 1,
 "overconfident": 31,
 "missed_information": 10,
 "applicability": 12,
 "invalid_output": 0,
 "evidence_missing": 0,
 "evidence_wrong": 16,
 "evidence_hallucinated": 3
}
13 cases written -> results/ft_ep1_val/error_analysis.md
saved /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/figures/training_curve.png
saved /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/figures/cm_baseline_qwen25_1p5b_test_overall.png
saved /kaggle/working/trial-eligibility-transformer/trial-eligibility-transformer/results/figures/cm_baseline_qwen25_1p5b_test_inclusion.png
saved /kaggle/working/trial-eligibility-transformer/trial

In [14]:
import json
for t in ['baseline_qwen25_1p5b', best, 'baseline_qwen25_1p5b_test', 'ft_best_test']:
    m = json.load(open(f'results/{t}/metrics.json'))
    o, i, e = m['overall'], m['inclusion'], m['exclusion']
    print(f"{t:28s} acc={o['classification']['accuracy']:.3f} mF1={o['classification']['macro_f1']:.3f} "
          f"incl={i['classification']['accuracy']:.3f} excl={e['classification']['accuracy']:.3f} "
          f"invalid={o['classification']['n_invalid']} evF1={o['evidence']['micro']['f1']:.3f}")
print(open('outputs/lora_qwen25_1p5b/train_config.json').read())

baseline_qwen25_1p5b         acc=0.189 mF1=0.143 incl=0.500 excl=0.043 invalid=0 evF1=0.185
ft_ep1_val                   acc=0.733 mF1=0.468 incl=0.742 excl=0.729 invalid=0 evF1=0.323
baseline_qwen25_1p5b_test    acc=0.170 mF1=0.156 incl=0.373 excl=0.053 invalid=0 evF1=0.203
ft_best_test                 acc=0.728 mF1=0.465 incl=0.787 excl=0.695 invalid=0 evF1=0.342
{
  "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
  "finetune": {
    "method": "lora",
    "output_dir": "outputs/lora_qwen25_1p5b",
    "max_seq_len": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "epochs": 3,
    "learning_rate": 0.0002,
    "per_device_batch_size": 1,
    "grad_accum": 16,
    "gradient_checkpointing": true,
    "warmup_ratio": 0.05,
    "lr_scheduler": "cosine",
    "weight_decay": 0.0,
    "seed": 42,
    "logging_steps": 5
  },

## 6. Download
`m2_results.zip` (results + figures + logs, small) and `lora_best_adapter.zip` (~70 MB adapter) from the Output panel.

In [15]:
!cp outputs/lora_qwen25_1p5b/train_log.json outputs/lora_qwen25_1p5b/train_config.json results/
!zip -qr /kaggle/working/m2_results.zip results data/splits
!cd outputs && zip -qr /kaggle/working/lora_best_adapter.zip lora_qwen25_1p5b_best
!ls -la /kaggle/working

total 67160
drwxr-xr-x  4 root root     4096 Nov  5 11:43 .
drwxr-xr-x  5 root root     4096 Nov  5 11:03 ..
-rw-r--r--  1 root root 68281274 Nov  5 11:43 lora_best_adapter.zip
-rw-r--r--  1 root root   467020 Nov  5 11:43 m2_results.zip
drwxr-xr-x 13 root root     4096 Nov  5 11:21 trial-eligibility-transformer
drwxr-xr-x  2 root root     4096 Nov  5 11:04 .virtual_documents
